### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** No senior architect feeds raw t-SNE/UMAP results into a downstream model. They destroy global metrics. For data engineering, we use clustering for **Target Leakage detection** and **Outlier Removal**. If your data doesn't naturally cluster, it's often a sign that your features are missing a critical causal signal.

**Mental Model:** 
- **K-Means:** A hard-boundary ruler. Every point belongs to exactly one centroid.
- **GMM:** A soft-colored lantern. Points near the center are 'mostly red', but points in between can be '30% red, 70% blue' (Soft Assignment).
- **Spectral Clustering:** Think of a graph of friendships. Clusters are defined by 'who knows who' (Similarity), not just physical location.

**Common Junior Pitfall:** Not scaling data before clustering. K-Means and GMM are distance/probability based. If one feature has scale [0, 1] and another [0, 1,000,000], the clustering will focus ONLY on the wide feature.

---


## 1. Centroid Clustering (K-Means)
K-Means tries to minimize **Inertia** (sum of squared distances to centroids). It is fast but only works for spherical clusters.

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Centroid Clustering (K-Means)](#1-centroid-clustering-k-means)
* [2. Gaussian Mixture Models — Probabilistic Soft Clustering](#2-gaussian-mixture-models-probabilistic-soft-clustering)
* [3. Spectral Clustering — Graph-Based Discovery](#3-spectral-clustering-graph-based-discovery)
* [4. UMAP — Topological Speed & Global Structure](#4-umap-topological-speed-global-structure)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: GMM assignment](#q1-gmm-assignment)
  * [Q2: Spectral Clustering's superpower](#q2-spectral-clusterings-superpower)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.6)
def simple_kmeans(X, k=3):
    centroids = X[np.random.choice(X.shape[0], k)]
    for _ in range(10):
        labels = np.argmin(np.linalg.norm(X[:, None] - centroids, axis=2), axis=1)
        centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
    return labels, centroids

labels, c = simple_kmeans(X)
plt.scatter(X[:, 0], X[:, 1], c=labels, s=30)
plt.scatter(c[:, 0], c[:, 1], marker='X', s=200, color='red')
plt.show()

"""
What just happened?
We built K-Means. Every point chooses its master (centroid) and every master 
moves to the average of its followers. This converges quickly to a local minimum.
"""

## 2. Gaussian Mixture Models — Probabilistic Soft Clustering

GMMs represent each cluster as a Gaussian $N(\mu_k, \Sigma_k)$. 

**The EM Algorithm:**
1. **E-Step (Expectation):** Calculate the probability $r_{ik}$ (Responsibility) of every point belonging to every cluster using Bayes' Theorem.
2. **M-Step (Maximization):** Update the means and covariances as the weighted average of all points, weighted by their $r_{ik}$ values.

🎓 **Socratic Deep Check:**
> *"If you set the covariance matrix $\Sigma$ of every GMM cluster to be the Identity matrix ($I$), and force the soft probabilities to be hard (0 or 1), why does the GMM become exactly identical to K-Means?"*

In [ ]:
from scipy.stats import multivariate_normal

def gmm_em(X, k=2, iters=50):
    n, d = X.shape
    mu = X[np.random.choice(n, k)]
    sigma = [np.eye(d)] * k
    pi = np.ones(k) / k
    r = np.zeros((n, k))
    
    for _ in range(iters):
        # E-step
        for j in range(k):
            r[:, j] = pi[j] * multivariate_normal.pdf(X, mu[j], sigma[j])
        r /= r.sum(axis=1, keepdims=True)
        
        # M-step
        for j in range(k):
            nj = r[:, j].sum()
            mu[j] = (r[:, j, None] * X).sum(axis=0) / nj
            sigma[j] = ((r[:, j, None] * (X - mu[j])).T @ (X - mu[j])) / nj
            pi[j] = nj / n
    return np.argmax(r, axis=1)

# Elongated data
rng = np.random.RandomState(42)
X_stretched = np.dot(rng.randn(300, 2), [[0.60, -0.70], [-0.20, 0.80]])
labels_gmm = gmm_em(X_stretched)
plt.scatter(X_stretched[:, 0], X_stretched[:, 1], c=labels_gmm, s=30)
plt.title("GMM Soft Clustering Handling Stretched Shapes")
plt.show()

"""
What just happened?
We implemented GMM. Unlike K-Means, these clusters 'learned' that they are 
elongated by calculating their covariance. Every point now has a probability 
distribution of belonging, not just a label.
"""

## 3. Spectral Clustering — Graph-Based Discovery

How do we cluster concentric rings where no center point exists? We look at the **Similarity Graph**.

1. **Adjacency Matrix $W$:** $W_{ij} = e^{-||x_i-x_j||^2 / \sigma^2}$ (High for close points).
2. **Graph Laplacian $L$:** $L = D - W$ (where $D$ is the degree diagonal).
3. **Eigen-decomposition:** Find the eigenvectors of $L$ with the smallest non-zero eigenvalues. This maps points into a space where they are linearly separable.


In [ ]:
from sklearn.datasets import make_circles
from scipy.linalg import eigh

X_circ, _ = make_circles(n_samples=200, factor=.5, noise=.05)

def spectral_clustering(X, k=2, sigma=0.1):
    # 1. Similarity matrix (RBF)
    sq_dists = np.sum((X[:, None] - X[None, :])**2, axis=2)
    W = np.exp(-sq_dists / (2 * sigma**2))
    
    # 2. Laplacian
    D = np.diag(W.sum(axis=1))
    L = D - W
    
    # 3. Spectral Embedding (K smallest eigenvectors)
    vals, vecs = eigh(L, subset_by_index=[0, k-1])
    
    # 4. K-Means on embedding
    from sklearn.cluster import KMeans
    return KMeans(n_clusters=k).fit_predict(vecs)

labels_sc = spectral_clustering(X_circ)
plt.scatter(X_circ[:, 0], X_circ[:, 1], c=labels_sc, s=30)
plt.title("Spectral Clustering Dividing Concentric Rings")
plt.show()

"""
What just happened?
We used graph theory. The Laplacian's eigenvectors find the 'low energy' cuts 
in the graph. Even though the rings are entwined, the graph sees them as 
disconnected components. K-Means would fail here; Spectral Clustering wins.
"""

## 4. UMAP — Topological Speed & Global Structure

**t-SNE** is the traditional king of visualization, but it is slow and destroys global distances. 
**UMAP (Uniform Manifold Approximation and Projection)** builds a fuzzy topological simplicial complex and optimizes a low-D layout that satisfies these connectivity rules.

**Advantages:**
1. **Speed:** Often 10x faster than t-SNE using Stochastic Gradient Descent.
2. **Structure:** Preserves global clusters much better than the neighborhood-obsessed t-SNE.
3. **Inference:** Unlike t-SNE, UMAP can transform new, unseen points.


In [ ]:
import time
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
try:
    from umap import UMAP
except ImportError:
    print("Installing umap-learn...")
    !pip install -q umap-learn
    from umap import UMAP

digits = load_digits()
X_sub = digits.data[:1000]

start = time.time()
X_tsne = TSNE(random_state=42).fit_transform(X_sub)
time_tsne = time.time() - start

start = time.time()
X_umap = UMAP(random_state=42).fit_transform(X_sub)
time_umap = time.time() - start

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target[:1000], s=10, cmap='Spectral')
ax1.set_title(f"t-SNE (Time: {time_tsne:.2f}s)")
ax2.scatter(X_umap[:, 0], X_umap[:, 1], c=digits.target[:1000], s=10, cmap='Spectral')
ax2.set_title(f"UMAP (Time: {time_umap:.2f}s)")
plt.show()

"""
What just happened?
We benchmarked the two visualizers. Note UMAP's speed advantage. More importantly, 
look at the clusters — UMAP often maps similar digits (like 1s and 7s) closer together 
while t-SNE pushes everything into arbitrary bubbles.
"""

---
## ✅ Knowledge Check

### Q1: GMM assignment
<details><summary>👉 Answer</summary>
GMM uses 'Soft Assignment' (probabilities). Every point has a score for every cluster. In production, this is critical for detecting 'borderline' cases that don't clearly belong to any group.
</details>

### Q2: Spectral Clustering's superpower
<details><summary>👉 Answer</summary>
Spectral clustering uses the Graph Laplacian to discover non-convex clusters (like moons or rings). It avoids the K-Means 'Spherical Bias' but requires building an $O(N^2)$ similarity matrix, which is expensive for large data.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement **K-Means++** initialization and compare its final inertia against random initialization over 10 trials.
2. Modify the GMM code to initialize means randomly instead of using data points. Observe if convergence takes longer.

### Tier 2: Intermediate
1. **BIC-based Cluster Selection:** Use Scikit-learn's `GaussianMixture` and compute the BIC (Bayesian Information Criterion) for K in [2, 10]. Find the K that minimizes BIC.
2. **Sparse Laplacian:** For Spectral Clustering, instead of building the full similarity matrix, only keep edges for the top-5 nearest neighbors of each point. How does this affect calculation speed?

### Tier 3: Advanced
1. **EM from Scratch Debugger:** Implement GMM with 'Full' vs 'Diagonal' covariance constraints. Prove that using a diagonal covariance forces the GMM to create clusters aligned ONLY with the axes.
2. **UMAP out-of-sample:** Use UMAP to fit an embedding for Digits [0-7]. Then, use the learned `.transform()` function to project Digits [8-9] into the same space. Verify if 8s and 9s land in logical locations near 0s or 1s.
